# ECS 入门
在开始进行量化研究之前，我们先介绍如何构建一个示例程序，并且在 ECS 中运行。

### 定义参数

首先，我们给这个示例程序定义一些环境变量：

In [ ]:
import boto3

aws_account_id = boto3.client('sts').get_caller_identity().get('Account')
repository_name = 'ecs-demo-php-simple-app'
aws_region = 'ap-southeast-1'

### 创建镜像仓库和容器镜像

接下来，我们使用 SDK 为示例程序创建镜像仓库：

In [ ]:
ecs = boto3.client('ecs', region_name=aws_region)
ecr = boto3.client('ecr', region_name=aws_region)

In [ ]:
ecr.create_repository(repositoryName=repository_name)

创建好镜像仓库之后，运行以下代码下载示例程序：

In [ ]:
!git clone https://github.com/aws-samples/ecs-demo-php-simple-app

构建镜像并打好 tag：

In [ ]:
!docker build ecs-demo-php-simple-app -t ecs-demo-php-simple-app

In [ ]:
!docker tag ecs-demo-php-simple-app {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{repository_name}

可以通过以下命令确认容器镜像已经构建成功：

In [ ]:
!docker images

接下来，登录到 ECR 并将镜像推送到仓库：

In [ ]:
!aws ecr get-login-password | docker login --username AWS --password-stdin {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com

In [ ]:
!docker push {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{repository_name}

### 创建 ECR 任务定义

接下来，我们在 ECS 服务中创建与镜像相对应的 task definition：

In [ ]:
ecs.register_task_definition(
    family="algo-trading-workshop",
    volumes=[
        {
            "name": "my-vol",
            "host": {}
        }
    ],
    networkMode="awsvpc",
    executionRoleArn="arn:aws:iam::{}:role/ecsTaskExecutionRole".format(aws_account_id),
    containerDefinitions=[
        {
            "name": "simple-app",
            "image": "{}.dkr.ecr.{}.amazonaws.com/{}:latest".format(aws_account_id, aws_region, repository_name),
            "portMappings": [
                {
                    "containerPort": 80,
                    "hostPort": 80,
                    "protocol": "tcp"
                }
            ],
            "mountPoints": [
                {
                    "sourceVolume": "my-vol",
                    "containerPath": "/var/www/my-vol"
                }
            ],
            "entryPoint": [
                "/usr/sbin/apache2",
                "-D",
                "FOREGROUND"
            ],
            "essential": True
        },
        {
            "name": "busybox",
            "image": "busybox",
            "cpu": 10,
            "memory": 500,
            "volumesFrom": [
            {
              "sourceContainer": "simple-app"
            }
            ],
            "entryPoint": [
                "sh",
                "-c"
            ],
            "command": [
                "/bin/sh -c \"while true; do /bin/date > /var/www/my-vol/date; sleep 1; done\""
            ],
            "essential": False
        }
    ],
    requiresCompatibilities=["FARGATE"],
    memory="512",
    cpu="256"
)

### 运行示例程序

再接下来，复制上面的'taskDefinitionArn'，再到 CloudFormation Output 中找到公有子网的 id 和 http 安全组 id：

In [ ]:
task_definition = "arn:aws:ecs:ap-southeast-1:843493563478:task-definition/sample-app:1"
subnet1 = "subnet-0371bb9e91f347a76"
subnet2 = "subnet-082e5fcc4408b9b64"
subnet3 = "subnet-012806508bf0f9451"
sg = "sg-0c40f5184518cec2f"

完成上述步骤之后，就可以再 Fargate 中创建容器来运行示例程序了：

In [ ]:
ecs.run_task(
    cluster='algo-trading-workshop',
    enableExecuteCommand=False,
    group='family:algo-trading-workshop',
    launchType='FARGATE',
    count=2,
    networkConfiguration={
                'awsvpcConfiguration': {
                    'subnets': [subnet1, subnet2, subnet3],
                    'securityGroups': [sg],
                    'assignPublicIp': 'ENABLED'
                }
            },
    taskDefinition=task_definition
)

最后再找到这个 task 相应的 公网ip，就可以访问这个示例 PHP 服务了。确认服务能够正常运行就可以把停止 task 了。